# W1 Day6（周六）：CNN + ResNet + ViT

**日期**: 04/12 | **时长**: 5h | **产出物**: ResNet+ViT实现 + CIFAR-10训练对比

---

## 🗺️ 今日路线图

| 模块 | 内容 | 时长 |
|------|------|------|
| 1 | 卷积基础：Conv2d 手写 + 参数计算 | 30min |
| 2 | 感受野：理论推导 + 可视化验证 | 20min |
| 3 | 经典 CNN 架构演进 (LeNet → VGG 思想) | 20min |
| 4 | ResNet：残差学习 + 恒等映射原理 | 30min |
| 5 | 手写 ResNet-50（完整实现 + 验证） | 40min |
| 6 | ViT：Patch Embedding + Positional Encoding | 30min |
| 7 | ViT：CLS Token + Transformer Encoder + 分类头 | 30min |
| 8 | 手写完整 ViT（从零实现） | 30min |
| 9 | CIFAR-10 训练对比：ResNet vs ViT | 40min |
| 10 | 可视化分析 + 总结 | 30min |

**前置依赖**: W1 Day3 (Tensor+Autograd), Day4 (Module+优化器+AMP)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import math
from collections import OrderedDict

# 全局设置
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 固定随机种子
torch.manual_seed(42)
np.random.seed(42)

---
## 模块 1：卷积基础 — 手写 Conv2d + 参数计算 (30min)

### 1.1 卷积运算的本质

卷积 ≠ 数学上的卷积（没有翻转），准确说是 **互相关 (cross-correlation)**。

**核心思想**: 局部连接 + 权重共享 → 大幅减少参数量

对于输入 $X \in \mathbb{R}^{C_{in} \times H \times W}$，卷积核 $W \in \mathbb{R}^{C_{out} \times C_{in} \times k_H \times k_W}$：

$$Y[c_{out}, i, j] = \sum_{c_{in}} \sum_{p} \sum_{q} X[c_{in}, i \cdot s + p, j \cdot s + q] \cdot W[c_{out}, c_{in}, p, q] + b[c_{out}]$$

其中 $s$ 是 stride。

In [ ]:
# ============================================================
# 实验 1.1: 手写 Conv2d（纯 Python/NumPy 实现）
# ============================================================

def conv2d_naive(input_tensor, weight, bias=None, stride=1, padding=0):
    """
    手写 2D 卷积 (支持 batch)
    input_tensor: (N, C_in, H, W)
    weight: (C_out, C_in, kH, kW)
    bias: (C_out,) or None
    """
    N, C_in, H, W = input_tensor.shape
    C_out, C_in_w, kH, kW = weight.shape
    assert C_in == C_in_w, f"Channel mismatch: {C_in} vs {C_in_w}"
    
    # padding
    if padding > 0:
        input_tensor = np.pad(
            input_tensor,
            ((0,0), (0,0), (padding,padding), (padding,padding)),
            mode='constant'
        )
    
    _, _, H_pad, W_pad = input_tensor.shape
    H_out = (H_pad - kH) // stride + 1
    W_out = (W_pad - kW) // stride + 1
    
    output = np.zeros((N, C_out, H_out, W_out))
    
    for n in range(N):           # batch
        for co in range(C_out):  # 输出通道
            for i in range(H_out):
                for j in range(W_out):
                    h_start = i * stride
                    w_start = j * stride
                    # 提取感受野区域 (C_in, kH, kW)
                    patch = input_tensor[n, :, h_start:h_start+kH, w_start:w_start+kW]
                    # 逐元素乘 + 求和
                    output[n, co, i, j] = np.sum(patch * weight[co])
            if bias is not None:
                output[n, co] += bias[co]
    
    return output

# 验证：与 PyTorch 对比
torch.manual_seed(42)
x = torch.randn(2, 3, 8, 8)  # batch=2, 3通道, 8x8
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=True)

# PyTorch 结果
y_pytorch = conv(x)

# 手写结果
y_naive = conv2d_naive(
    x.numpy(),
    conv.weight.detach().numpy(),
    conv.bias.detach().numpy(),
    stride=1, padding=1
)

diff = np.max(np.abs(y_pytorch.detach().numpy() - y_naive))
print(f"手写 Conv2d vs PyTorch 最大误差: {diff:.2e}")
print(f"✅ 验证通过！" if diff < 1e-5 else "❌ 误差过大")
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {y_pytorch.shape}")

In [ ]:
# ============================================================
# 实验 1.2: 卷积参数量计算 + 与全连接对比
# ============================================================

def count_conv_params(C_in, C_out, kernel_size, bias=True):
    """计算卷积层参数量"""
    k = kernel_size if isinstance(kernel_size, int) else kernel_size[0] * kernel_size[1]
    if isinstance(kernel_size, int):
        k = kernel_size * kernel_size
    params = C_out * C_in * k
    if bias:
        params += C_out
    return params

# 场景：处理 224x224 RGB 图像，输出 64 通道特征图
H, W, C_in, C_out = 224, 224, 3, 64

conv_params = count_conv_params(C_in, C_out, 3)
fc_params = H * W * C_in * H * W * C_out  # 全连接（每个输出连所有输入）
# 更合理的比较：全连接将图像展平后映射到同样大小的输出
fc_params_flat = (H * W * C_in) * (H * W * C_out)

print("=" * 60)
print("卷积 vs 全连接 参数量对比")
print("=" * 60)
print(f"任务: {C_in}×{H}×{W} → {C_out} 通道特征图")
print(f"Conv2d(3, 64, kernel_size=3): {conv_params:>15,} 参数")
print(f"全连接等价:                   {fc_params_flat:>15,} 参数")
print(f"参数量减少比例:               {fc_params_flat/conv_params:>15,.0f}x")
print()

# 验证参数量计算
conv_layer = nn.Conv2d(3, 64, 3, padding=1)
actual = sum(p.numel() for p in conv_layer.parameters())
print(f"PyTorch 实际参数量: {actual} (我们计算的: {conv_params})")
print(f"✅ 一致！" if actual == conv_params else "❌ 不一致")

In [ ]:
# ============================================================
# 实验 1.3: 输出尺寸公式 + 各种配置实验
# ============================================================

def output_size(H, W, kernel_size, stride=1, padding=0, dilation=1):
    """计算卷积输出尺寸: H_out = floor((H + 2*padding - dilation*(kernel-1) - 1) / stride + 1)"""
    k = kernel_size
    H_out = math.floor((H + 2*padding - dilation*(k-1) - 1) / stride + 1)
    W_out = math.floor((W + 2*padding - dilation*(k-1) - 1) / stride + 1)
    return H_out, W_out

print("输出尺寸公式: H_out = floor((H + 2p - d(k-1) - 1) / s + 1)")
print("=" * 70)

configs = [
    {"name": "基本 3x3",       "H": 32, "W": 32, "k": 3, "s": 1, "p": 0, "d": 1},
    {"name": "3x3 + padding=1", "H": 32, "W": 32, "k": 3, "s": 1, "p": 1, "d": 1},
    {"name": "3x3 + stride=2",  "H": 32, "W": 32, "k": 3, "s": 2, "p": 1, "d": 1},
    {"name": "5x5 + padding=2", "H": 32, "W": 32, "k": 5, "s": 1, "p": 2, "d": 1},
    {"name": "3x3 dilation=2",  "H": 32, "W": 32, "k": 3, "s": 1, "p": 2, "d": 2},
    {"name": "7x7 stride=2 p=3","H": 224,"W": 224,"k": 7, "s": 2, "p": 3, "d": 1},
]

for cfg in configs:
    H_out, W_out = output_size(cfg['H'], cfg['W'], cfg['k'], cfg['s'], cfg['p'], cfg['d'])
    # PyTorch 验证
    conv = nn.Conv2d(1, 1, cfg['k'], stride=cfg['s'], padding=cfg['p'], dilation=cfg['d'])
    x = torch.randn(1, 1, cfg['H'], cfg['W'])
    y = conv(x)
    actual_H, actual_W = y.shape[2], y.shape[3]
    match = "✅" if (H_out == actual_H and W_out == actual_W) else "❌"
    print(f"{cfg['name']:20s}: {cfg['H']}×{cfg['W']} → {H_out}×{W_out}  "
          f"(k={cfg['k']}, s={cfg['s']}, p={cfg['p']}, d={cfg['d']}) {match}")

print("\n💡 关键记忆:")
print("  • 3×3 conv + padding=1 + stride=1 → 尺寸不变（最常用）")
print("  • stride=2 → 尺寸减半（替代 pooling 的下采样方式）")
print("  • 7×7 stride=2 padding=3 → 224→112（ResNet 第一层）")

---
## 模块 2：感受野 — 理论推导 + 可视化验证 (20min)

### 2.1 感受野 (Receptive Field) 的定义

**感受野**: 输出特征图上一个神经元"看到"的输入图像区域大小。

**递推公式** (从后往前推):

$$RF_l = RF_{l+1} + (k_l - 1) \times \prod_{i=l+1}^{L} s_i$$

简化版（从前往后推）:
- $RF_0 = 1$（输入层每个像素的感受野 = 1）
- $RF_l = RF_{l-1} + (k_l - 1) \times \text{jump}_{l-1}$
- $\text{jump}_l = \text{jump}_{l-1} \times s_l$

In [ ]:
# ============================================================
# 实验 2.1: 感受野递推计算
# ============================================================

def compute_receptive_field(layers):
    """
    计算每一层的感受野
    layers: [(name, kernel_size, stride), ...]
    返回: [(name, rf, jump), ...]
    """
    rf = 1      # 初始感受野
    jump = 1    # 初始跳跃 (stride 的累积)
    results = [("Input", rf, jump)]
    
    for name, k, s in layers:
        rf = rf + (k - 1) * jump
        jump = jump * s
        results.append((name, rf, jump))
    
    return results

# 示例 1: 三层 3x3 卷积
simple_layers = [
    ("Conv3x3_1", 3, 1),
    ("Conv3x3_2", 3, 1),
    ("Conv3x3_3", 3, 1),
]

print("=" * 50)
print("三层 3×3 卷积的感受野")
print("=" * 50)
for name, rf, jump in compute_receptive_field(simple_layers):
    print(f"  {name:12s}: RF = {rf:3d}, Jump = {jump}")

print("\n💡 三层 3×3 的感受野 = 7×7，等效于一层 7×7 卷积！")
print("   但参数量: 3×(3²C²) = 27C² vs 7²C² = 49C²，减少 45%")
print("   且三层有三次非线性！（VGGNet 的核心洞察）")

# 示例 2: VGG-16 前几层
print("\n" + "=" * 50)
print("VGG-16 风格的感受野增长")
print("=" * 50)
vgg_layers = [
    ("Conv3x3", 3, 1), ("Conv3x3", 3, 1), ("MaxPool", 2, 2),
    ("Conv3x3", 3, 1), ("Conv3x3", 3, 1), ("MaxPool", 2, 2),
    ("Conv3x3", 3, 1), ("Conv3x3", 3, 1), ("Conv3x3", 3, 1), ("MaxPool", 2, 2),
]
for name, rf, jump in compute_receptive_field(vgg_layers):
    print(f"  {name:12s}: RF = {rf:3d}, Jump = {jump}")

# 示例 3: ResNet 风格
print("\n" + "=" * 50)
print("ResNet 风格的感受野增长")
print("=" * 50)
resnet_layers = [
    ("Conv7x7_s2", 7, 2),  # stem
    ("MaxPool3x3_s2", 3, 2),
    # stage 1 (3 bottleneck blocks)
    ("Conv1x1", 1, 1), ("Conv3x3", 3, 1), ("Conv1x1", 1, 1),
    ("Conv1x1", 1, 1), ("Conv3x3", 3, 1), ("Conv1x1", 1, 1),
    ("Conv1x1", 1, 1), ("Conv3x3", 3, 1), ("Conv1x1", 1, 1),
    # stage 2 first block (stride=2 in 3x3)
    ("Conv1x1", 1, 1), ("Conv3x3_s2", 3, 2), ("Conv1x1", 1, 1),
]
for name, rf, jump in compute_receptive_field(resnet_layers):
    print(f"  {name:12s}: RF = {rf:3d}, Jump = {jump}")

In [ ]:
# ============================================================
# 实验 2.2: 感受野的梯度验证（反向传播法）
# ============================================================

def verify_receptive_field_by_gradient(model, input_size=(1, 3, 32, 32)):
    """
    用梯度反向传播验证实际感受野
    原理：选输出的中心像素，backward，看输入哪些位置梯度≠0
    """
    x = torch.randn(*input_size, requires_grad=True)
    y = model(x)
    
    # 选择输出的中心点
    _, _, H_out, W_out = y.shape
    center_h, center_w = H_out // 2, W_out // 2
    
    # 只对中心点 backward
    y[0, 0, center_h, center_w].backward()
    
    # 梯度不为 0 的区域就是实际感受野
    grad = x.grad[0, 0].abs()  # 取第一个通道
    nonzero_mask = (grad > 0).float()
    
    # 找边界
    nonzero_coords = torch.nonzero(nonzero_mask)
    if len(nonzero_coords) > 0:
        h_min, w_min = nonzero_coords.min(dim=0).values
        h_max, w_max = nonzero_coords.max(dim=0).values
        rf_h = (h_max - h_min + 1).item()
        rf_w = (w_max - w_min + 1).item()
    else:
        rf_h = rf_w = 0
    
    return rf_h, rf_w, grad.numpy()

# 三层 3x3 卷积
model_3x3 = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 1, 3, padding=1),
)

rf_h, rf_w, grad_map = verify_receptive_field_by_gradient(model_3x3)
print(f"三层 3×3 卷积的实际感受野 (梯度法): {rf_h}×{rf_w}")
print(f"理论感受野: 7×7")
print(f"✅ 一致！" if rf_h == 7 else "❌ 不一致")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 梯度热力图
im = axes[0].imshow(grad_map, cmap='hot', interpolation='nearest')
axes[0].set_title(f'Gradient Map (RF = {rf_h}×{rf_w})', fontsize=14)
plt.colorbar(im, ax=axes[0])

# 中心区域放大
center = 16
margin = 5
zoomed = grad_map[center-margin:center+margin+1, center-margin:center+margin+1]
im2 = axes[1].imshow(zoomed, cmap='hot', interpolation='nearest')
axes[1].set_title('Zoomed Center (gradient magnitude)', fontsize=14)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n💡 注意: 虽然理论感受野 = 7×7，但中心区域梯度更大")
print("   → '有效感受野' 远小于理论感受野（这是一个重要的研究发现）")

---
## 模块 3：经典 CNN 架构演进 (20min)

### 3.1 从 LeNet 到 VGG 的演进

| 架构 | 年份 | 核心贡献 | 深度 | Top-5 错误率 |
|------|------|----------|------|-------------|
| LeNet-5 | 1998 | 卷积+池化基本范式 | 5 | - |
| AlexNet | 2012 | ReLU + Dropout + GPU训练 | 8 | 16.4% |
| VGGNet | 2014 | 小卷积核堆叠 (3×3) | 16/19 | 7.3% |
| GoogLeNet | 2014 | Inception模块(多尺度并行) | 22 | 6.7% |
| **ResNet** | **2015** | **残差连接** | **152** | **3.57%** |

### 关键问题：为什么不能简单地堆更多层？

**退化问题** (Degradation Problem): 更深的网络反而训练误差更高！

- 不是过拟合（训练集上就更差了）
- 不完全是梯度消失（BN 已经缓解）
- 是优化困难：深层网络更难学习恒等映射

In [ ]:
# ============================================================
# 实验 3.1: 演示退化问题
# ============================================================

class PlainBlock(nn.Module):
    """无残差的基本块"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out)  # 注意：没有残差连接

class PlainNet(nn.Module):
    """Plain Network (无残差)"""
    def __init__(self, num_blocks, channels=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )
        self.blocks = nn.Sequential(*[PlainBlock(channels) for _ in range(num_blocks)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, 10)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

# 观察不同深度的梯度情况
for n_blocks in [5, 20, 50]:
    model = PlainNet(n_blocks, channels=32)
    x = torch.randn(4, 3, 32, 32)
    y = model(x)
    loss = y.sum()
    loss.backward()
    
    # 检查第一个 block 的梯度范数
    first_grad = model.blocks[0].conv1.weight.grad.norm().item()
    last_grad = model.blocks[-1].conv1.weight.grad.norm().item()
    n_params = sum(p.numel() for p in model.parameters())
    
    print(f"PlainNet ({n_blocks:2d} blocks, {n_params/1e3:.0f}K params): "
          f"第一层梯度={first_grad:.6f}, 最后层梯度={last_grad:.6f}, "
          f"比率={last_grad/first_grad:.2f}x")

print("\n💡 观察: 随着深度增加，第一层梯度变得非常小 → 训练困难")

---
## 模块 4：ResNet — 残差学习 + 恒等映射原理 (30min)

### 4.1 残差学习的核心思想

**问题**: 直接学习 $H(x)$ 很难

**解决**: 改为学习残差 $F(x) = H(x) - x$，即 $H(x) = F(x) + x$

**为什么有效？**

1. **恒等映射容易学**: 如果某层多余，只需 $F(x) \to 0$（权重趋零），输出就是 $x$ 本身
2. **梯度直达**: $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial H} \cdot (1 + \frac{\partial F}{\partial x})$，其中 **1** 保证梯度至少为 1
3. **类似 LSTM 的门控**: skip connection 类似 LSTM 中的遗忘门

### 4.2 两种 Residual Block

| | Basic Block | Bottleneck Block |
|---|---|---|
| 结构 | 3×3 → 3×3 | 1×1 → 3×3 → 1×1 |
| 用于 | ResNet-18/34 | ResNet-50/101/152 |
| 计算量 | 2 × 3²C² = 18C² | C²/4 + 9C²/16 + C²/4 ≈ 比基本块少 |
| 设计 | 直接处理 | 先压缩→处理→扩展（瓶颈） |

In [ ]:
# ============================================================
# 实验 4.1: 残差连接的梯度优势
# ============================================================

class ResBlock(nn.Module):
    """带残差连接的基本块"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x):
        identity = x  # 恒等映射！
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity  # 残差连接！
        return F.relu(out)

class ResNet_Simple(nn.Module):
    """简单 ResNet (用于对比)"""
    def __init__(self, num_blocks, channels=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )
        self.blocks = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, 10)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

# 对比 Plain vs Residual 的梯度
print("=" * 70)
print("PlainNet vs ResNet 梯度对比")
print("=" * 70)

for n_blocks in [5, 20, 50]:
    # Plain
    plain = PlainNet(n_blocks, channels=32)
    x = torch.randn(4, 3, 32, 32)
    plain(x).sum().backward()
    plain_first = plain.blocks[0].conv1.weight.grad.norm().item()
    
    # Residual
    resnet = ResNet_Simple(n_blocks, channels=32)
    x = torch.randn(4, 3, 32, 32)
    resnet(x).sum().backward()
    res_first = resnet.blocks[0].conv1.weight.grad.norm().item()
    
    print(f"  {n_blocks:2d} blocks: Plain第一层梯度 = {plain_first:.6f}, "
          f"ResNet第一层梯度 = {res_first:.6f}, "
          f"比值 = {res_first/plain_first:.1f}x")

print("\n💡 ResNet 的第一层梯度始终保持较大 → 深层网络也能有效训练！")

In [ ]:
# ============================================================
# 实验 4.2: 初始化时残差分支输出接近 0 的验证
# ============================================================

block = ResBlock(64)
x = torch.randn(1, 64, 32, 32)

with torch.no_grad():
    # 看残差分支 F(x) 的输出
    out = block.bn2(block.conv2(F.relu(block.bn1(block.conv1(x)))))
    identity = x
    
    f_norm = out.norm().item()
    x_norm = identity.norm().item()
    ratio = f_norm / x_norm

print(f"||F(x)|| = {f_norm:.4f}")
print(f"||x||    = {x_norm:.4f}")
print(f"||F(x)|| / ||x|| = {ratio:.4f}")
print(f"\n💡 初始化时 BN 使 F(x) ≈ 0，所以 H(x) = F(x) + x ≈ x")
print("   → 网络初始时接近恒等映射，训练从这里开始慢慢学习残差")

---
## 模块 5：手写 ResNet-50（完整实现 + 验证） (40min)

### ResNet-50 架构

```
输入: 3×224×224
  │
  ├── Stem: Conv 7×7, s=2 → BN → ReLU → MaxPool 3×3 s=2  →  64×56×56
  │
  ├── Stage 1 (conv2_x): 3 × Bottleneck(64, 256)           → 256×56×56
  ├── Stage 2 (conv3_x): 4 × Bottleneck(128, 512), s=2     → 512×28×28
  ├── Stage 3 (conv4_x): 6 × Bottleneck(256, 1024), s=2    → 1024×14×14
  ├── Stage 4 (conv5_x): 3 × Bottleneck(512, 2048), s=2    → 2048×7×7
  │
  ├── AdaptiveAvgPool → 2048×1×1
  └── FC → 1000
```

In [ ]:
# ============================================================
# 实验 5.1: 手写 Bottleneck Block
# ============================================================

class Bottleneck(nn.Module):
    """
    ResNet Bottleneck Block
    
    1×1 conv (降维) → 3×3 conv → 1×1 conv (升维)
         + shortcut (可能有 1×1 conv 调整维度)
    
    expansion = 4: 输出通道数 = planes * 4
    例如 planes=64 → 64→64→256
    """
    expansion = 4
    
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        
        # 1×1: 降维 in_planes → planes
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        
        # 3×3: 空间处理 (可能 stride=2 下采样)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        
        # 1×1: 升维 planes → planes * 4
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        
        self.downsample = downsample  # shortcut 维度匹配
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = x
        
        # 主路径
        out = self.relu(self.bn1(self.conv1(x)))     # 1×1 降维
        out = self.relu(self.bn2(self.conv2(out)))    # 3×3 空间
        out = self.bn3(self.conv3(out))               # 1×1 升维 (注意: 这里不加 ReLU)
        
        # shortcut 路径
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out += identity  # 残差相加
        out = self.relu(out)  # 加完再 ReLU
        
        return out

# 验证单个 Bottleneck
# Case 1: 无下采样 (in=256, planes=64, out=256)
block1 = Bottleneck(256, 64)
x1 = torch.randn(1, 256, 56, 56)
y1 = block1(x1)
print(f"无下采样: {x1.shape} → {y1.shape}")

# Case 2: 有下采样 (in=256, planes=128, stride=2, out=512)
downsample = nn.Sequential(
    nn.Conv2d(256, 512, 1, stride=2, bias=False),
    nn.BatchNorm2d(512)
)
block2 = Bottleneck(256, 128, stride=2, downsample=downsample)
x2 = torch.randn(1, 256, 56, 56)
y2 = block2(x2)
print(f"有下采样: {x2.shape} → {y2.shape}")

In [ ]:
# ============================================================
# 实验 5.2: 手写完整 ResNet
# ============================================================

class ResNet(nn.Module):
    """
    完整 ResNet 实现
    支持 ResNet-50/101/152
    """
    def __init__(self, block, layers, num_classes=1000):
        """
        block: Bottleneck 类
        layers: 每个 stage 的 block 数量, e.g., [3, 4, 6, 3] for ResNet-50
        """
        super().__init__()
        self.in_planes = 64  # 追踪当前通道数
        
        # ---- Stem ----
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # ---- 4 个 Stage ----
        self.layer1 = self._make_layer(block, 64,  layers[0], stride=1)  # 56×56
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)  # 28×28
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)  # 14×14
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)  #  7×7
        
        # ---- 分类头 ----
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        
        # ---- 初始化 ----
        self._initialize_weights()
    
    def _make_layer(self, block, planes, num_blocks, stride):
        """
        创建一个 stage
        第一个 block 可能需要 downsample（维度/空间不匹配）
        后续 block 直接残差连接
        """
        downsample = None
        
        # 需要 downsample 的两种情况:
        # 1. stride != 1 (空间下采样)
        # 2. in_planes != planes * expansion (通道数不匹配)
        if stride != 1 or self.in_planes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes * block.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion)
            )
        
        layers = []
        # 第一个 block (可能 stride=2, 可能 downsample)
        layers.append(block(self.in_planes, planes, stride, downsample))
        self.in_planes = planes * block.expansion
        
        # 后续 block (stride=1, 无 downsample)
        for _ in range(1, num_blocks):
            layers.append(block(self.in_planes, planes))
        
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        """Kaiming 初始化"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Stem: 224→112→56
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        
        # 4 个 Stage
        x = self.layer1(x)  # 56→56
        x = self.layer2(x)  # 56→28
        x = self.layer3(x)  # 28→14
        x = self.layer4(x)  # 14→7
        
        # 分类
        x = self.avgpool(x)  # 7→1
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ---- 工厂函数 ----
def resnet50(num_classes=1000):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

def resnet101(num_classes=1000):
    return ResNet(Bottleneck, [3, 4, 23, 3], num_classes)

def resnet152(num_classes=1000):
    return ResNet(Bottleneck, [3, 8, 36, 3], num_classes)

In [ ]:
# ============================================================
# 实验 5.3: 验证 ResNet-50 + 各层特征图尺寸
# ============================================================

model = resnet50(num_classes=1000)

# 用 hook 记录每个 stage 的输出尺寸
feature_sizes = {}

def make_hook(name):
    def hook(module, input, output):
        feature_sizes[name] = output.shape
    return hook

model.conv1.register_forward_hook(make_hook('stem_conv'))
model.maxpool.register_forward_hook(make_hook('stem_maxpool'))
model.layer1.register_forward_hook(make_hook('stage1'))
model.layer2.register_forward_hook(make_hook('stage2'))
model.layer3.register_forward_hook(make_hook('stage3'))
model.layer4.register_forward_hook(make_hook('stage4'))
model.avgpool.register_forward_hook(make_hook('avgpool'))

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    y = model(x)

print("ResNet-50 各层特征图尺寸:")
print("=" * 55)
print(f"  {'层':15s} {'形状':25s} {'说明'}")
print("-" * 55)
print(f"  {'Input':15s} {str(tuple(x.shape)):25s}")
for name, shape in feature_sizes.items():
    notes = {
        'stem_conv': '7×7 s=2 → 尺寸减半',
        'stem_maxpool': '3×3 s=2 → 再减半',
        'stage1': '3 blocks, s=1',
        'stage2': '4 blocks, s=2',
        'stage3': '6 blocks, s=2',
        'stage4': '3 blocks, s=2',
        'avgpool': '全局平均池化',
    }
    print(f"  {name:15s} {str(tuple(shape)):25s} {notes.get(name, '')}")
print(f"  {'Output':15s} {str(tuple(y.shape)):25s} 1000 类")

# 参数量统计
total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params:,} ({total_params/1e6:.1f}M)")

# 与官方对比
official = torchvision.models.resnet50(weights=None)
official_params = sum(p.numel() for p in official.parameters())
print(f"官方 ResNet-50 参数量: {official_params:,}")
print(f"✅ 一致！" if total_params == official_params else f"❌ 差异: {total_params - official_params}")

In [ ]:
# ============================================================
# 实验 5.4: 与 torchvision ResNet-50 权重对齐验证
# ============================================================

# 将官方模型的权重加载到我们的模型
our_model = resnet50(num_classes=1000)
official_model = torchvision.models.resnet50(weights=None)

# 对比 state_dict 的 key
our_keys = set(our_model.state_dict().keys())
official_keys = set(official_model.state_dict().keys())

print(f"我们的模型 key 数量: {len(our_keys)}")
print(f"官方模型 key 数量:   {len(official_keys)}")

# 检查是否可以互相加载
try:
    our_model.load_state_dict(official_model.state_dict())
    print("✅ 成功加载官方权重到我们的模型！架构完全一致")
    
    # 前向传播对比
    x = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        y_ours = our_model(x)
        y_official = official_model(x)
    
    diff = (y_ours - y_official).abs().max().item()
    print(f"前向传播最大误差: {diff:.2e}")
    print(f"✅ 输出完全一致！" if diff < 1e-5 else "⚠️ 有微小差异")
    
except RuntimeError as e:
    print(f"❌ 加载失败: {e}")
    # 找出不同的 key
    only_ours = our_keys - official_keys
    only_official = official_keys - our_keys
    if only_ours:
        print(f"  我们独有: {only_ours}")
    if only_official:
        print(f"  官方独有: {only_official}")

In [ ]:
# ============================================================
# 实验 5.5: ResNet 各 stage 参数量 + FLOPs 分布
# ============================================================

model = resnet50()

# 统计每个 stage 的参数量
stage_params = {}
for name, child in model.named_children():
    params = sum(p.numel() for p in child.parameters())
    if params > 0:
        stage_params[name] = params

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 参数量分布
names = list(stage_params.keys())
values = list(stage_params.values())
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))
bars = axes[0].bar(names, [v/1e6 for v in values], color=colors)
axes[0].set_ylabel('Parameters (M)')
axes[0].set_title('ResNet-50 Parameters per Stage')
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val/1e6:.1f}M', ha='center', va='bottom', fontsize=9)

# 饼图
axes[1].pie([v for v in values], labels=names, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Parameter Distribution')

plt.tight_layout()
plt.show()

print(f"\n💡 观察:")
print(f"  • fc 层只占很少参数（不像 VGG 的 FC 占了 90%）")
print(f"  • layer4 (512→2048) 参数最多，因为通道数最大")
print(f"  • 参数量主要在 1×1 conv（通道变换），不在 3×3 conv")

---
## 模块 6：ViT — Patch Embedding + Positional Encoding (30min)

### Vision Transformer 核心思想

**"An Image is Worth 16x16 Words"** — 把图像切成 patch，每个 patch 当成一个 "token"

关键步骤:
1. **Patch Embedding**: 将图像切成 N 个 patch → 线性投影到 D 维
2. **CLS Token**: 可学习的分类 token，拼接在序列前面
3. **Positional Encoding**: 可学习的位置编码
4. **Transformer Encoder**: 标准的多头自注意力 + FFN
5. **分类头**: 取 CLS token 的输出做分类

In [ ]:
# ============================================================
# 实验 6.1: Patch Embedding — 两种实现方式
# ============================================================

class PatchEmbedding_Manual(nn.Module):
    """方式 1: 手动切 patch + 线性投影"""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size * in_channels, embed_dim)
    
    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        
        # 切 patch: (B, C, H, W) → (B, C, H/p, p, W/p, p)
        x = x.reshape(B, C, H//p, p, W//p, p)
        # 调整顺序: → (B, H/p, W/p, C, p, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        # 展平 patch: → (B, num_patches, C*p*p)
        x = x.reshape(B, self.num_patches, -1)
        # 线性投影: → (B, num_patches, embed_dim)
        x = self.proj(x)
        
        return x

class PatchEmbedding_Conv(nn.Module):
    """方式 2: 用卷积实现 (更高效！)"""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # 用 kernel=patch_size, stride=patch_size 的卷积
        # 等效于切 patch + 线性投影！
        self.proj = nn.Conv2d(in_channels, embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        # (B, C, H, W) → (B, embed_dim, H/p, W/p)
        x = self.proj(x)
        # → (B, embed_dim, num_patches) → (B, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        return x

# 验证两种方式等价
torch.manual_seed(42)
x = torch.randn(2, 3, 224, 224)

pe_manual = PatchEmbedding_Manual()
pe_conv = PatchEmbedding_Conv()

# 对齐权重
with torch.no_grad():
    # Conv2d.weight: (embed_dim, in_ch, p, p) → reshape 匹配 Linear
    pe_manual.proj.weight.copy_(
        pe_conv.proj.weight.reshape(768, -1)
    )
    pe_manual.proj.bias.copy_(pe_conv.proj.bias)

y_manual = pe_manual(x)
y_conv = pe_conv(x)

print(f"Manual Patch Embedding 输出: {y_manual.shape}")
print(f"Conv Patch Embedding 输出:   {y_conv.shape}")
print(f"最大误差: {(y_manual - y_conv).abs().max():.2e}")
print(f"✅ 两种实现等价！")

num_patches = (224 // 16) ** 2
print(f"\n图像 224×224, patch_size=16:")
print(f"  patch 数量: {num_patches} = (224/16)² = 14²")
print(f"  每个 patch: {16}×{16}×3 = {16*16*3} 维 → 投影到 768 维")

In [ ]:
# ============================================================
# 实验 6.2: 可视化 Patch 分割
# ============================================================

def visualize_patches(img_tensor, patch_size=16):
    """
    可视化图像如何被分割成 patch
    img_tensor: (C, H, W)
    """
    C, H, W = img_tensor.shape
    n_h = H // patch_size
    n_w = W // patch_size
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 原图 + 网格线
    img = img_tensor.permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())  # 归一化到 [0,1]
    
    axes[0].imshow(img)
    for i in range(n_h + 1):
        axes[0].axhline(y=i*patch_size, color='red', linewidth=0.5)
    for j in range(n_w + 1):
        axes[0].axvline(x=j*patch_size, color='red', linewidth=0.5)
    axes[0].set_title(f'Image with {n_h}×{n_w} = {n_h*n_w} patches (patch_size={patch_size})')
    
    # 展示部分 patch
    n_show = min(16, n_h * n_w)  # 展示前 16 个 patch
    grid_size = int(np.ceil(np.sqrt(n_show)))
    
    # 拼接成一个大图
    canvas = np.ones((grid_size * (patch_size+2), grid_size * (patch_size+2), 3))
    for idx in range(n_show):
        i, j = idx // n_w, idx % n_w
        patch = img[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size]
        
        row = idx // grid_size
        col = idx % grid_size
        r_start = row * (patch_size + 2) + 1
        c_start = col * (patch_size + 2) + 1
        canvas[r_start:r_start+patch_size, c_start:c_start+patch_size] = patch
    
    axes[1].imshow(canvas)
    axes[1].set_title(f'First {n_show} patches (each {patch_size}×{patch_size})')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# 用随机图像模拟（没有 ImageNet 数据）
# 创建一个有结构的测试图像
test_img = torch.zeros(3, 224, 224)
# 添加一些渐变和形状
for i in range(224):
    for j in range(224):
        test_img[0, i, j] = i / 224  # 红色渐变
        test_img[1, i, j] = j / 224  # 绿色渐变
        if (i - 112)**2 + (j - 112)**2 < 60**2:
            test_img[2, i, j] = 1.0  # 蓝色圆

visualize_patches(test_img, patch_size=16)

In [ ]:
# ============================================================
# 实验 6.3: CLS Token + Positional Encoding
# ============================================================

print("=" * 60)
print("CLS Token + Positional Encoding")
print("=" * 60)

embed_dim = 768
num_patches = 196  # 14×14
batch_size = 2

# CLS Token: 可学习的向量，代表整个图像
cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
print(f"CLS Token 形状: {cls_token.shape}")
print(f"  → 每个样本前面拼一个，序列长度 {num_patches} → {num_patches + 1}")

# Positional Encoding: 可学习的位置编码
pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim) * 0.02)
print(f"\nPosition Embedding 形状: {pos_embed.shape}")
print(f"  → 每个位置 (含 CLS) 有独立的位置编码")

# 模拟完整的 embedding 过程
x = torch.randn(batch_size, num_patches, embed_dim)  # patch embeddings
print(f"\n完整 Embedding 过程:")
print(f"  1. Patch Embedding: {x.shape}")

# 扩展 CLS token 到 batch
cls_tokens = cls_token.expand(batch_size, -1, -1)
print(f"  2. CLS Token 扩展: {cls_tokens.shape}")

# 拼接
x = torch.cat([cls_tokens, x], dim=1)
print(f"  3. 拼接后: {x.shape}  (1 CLS + {num_patches} patches)")

# 加位置编码
x = x + pos_embed
print(f"  4. 加位置编码后: {x.shape}")

print(f"\n💡 为什么用 CLS Token 而不用 Global Average Pooling?")
print(f"  • CLS Token 类似 BERT，从所有 patch 聚合全局信息")
print(f"  • 实际上后来的 ViT 变体发现 GAP 也行 (DeiT III)")
print(f"  • 但 CLS Token 更灵活，可以用于多种下游任务")

In [ ]:
# ============================================================
# 实验 6.4: 位置编码的可视化分析
# ============================================================

# 训练后的位置编码会学到什么？
# 模拟：计算位置编码之间的余弦相似度

def visualize_position_embeddings(pos_embed, num_patches_h=14):
    """
    可视化位置编码的相似度模式
    pos_embed: (1, num_patches+1, embed_dim)
    """
    # 去掉 CLS token 的位置编码
    patch_pos = pos_embed[0, 1:]  # (196, 768)
    
    # 计算余弦相似度矩阵
    patch_pos_norm = F.normalize(patch_pos, dim=-1)
    sim_matrix = torch.mm(patch_pos_norm, patch_pos_norm.t())  # (196, 196)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # 选择几个代表性位置，看它与所有位置的相似度
    positions = [
        (0, 0), (0, 7), (0, 13),  # 上边
        (7, 0), (7, 7),            # 中间
        (13, 0), (13, 7), (13, 13) # 下边
    ]
    
    for idx, (pi, pj) in enumerate(positions):
        ax = axes[idx // 4, idx % 4]
        pos_idx = pi * num_patches_h + pj
        sim = sim_matrix[pos_idx].reshape(num_patches_h, num_patches_h).detach().numpy()
        im = ax.imshow(sim, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.plot(pj, pi, 'k*', markersize=10)  # 标记查询位置
        ax.set_title(f'pos ({pi},{pj})', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    
    plt.suptitle('Position Embedding Cosine Similarity\n(random init - 训练后会呈现更明显的空间结构)', fontsize=14)
    plt.tight_layout()
    plt.show()

# 随机初始化的位置编码
torch.manual_seed(42)
pos_embed = nn.Parameter(torch.randn(1, 197, 768) * 0.02)
visualize_position_embeddings(pos_embed)

print("💡 随机初始化时没有明显空间结构")
print("   训练后，相邻 patch 的位置编码会变得更相似")
print("   形成类似 2D 坐标的结构（ViT 论文 Figure 7）")

---
## 模块 7：ViT 的 Transformer Encoder + 分类头 (30min)

### ViT 使用的 Transformer Encoder

与标准 Transformer 的两个区别:
1. **Pre-Norm** (Layer Norm 在 Attention/FFN 之前)
2. 没有 Decoder（纯 Encoder）

In [ ]:
# ============================================================
# 实验 7.1: ViT 中的 Multi-Head Self-Attention
# ============================================================

class MultiHeadSelfAttention(nn.Module):
    """ViT 的 MHSA 实现"""
    def __init__(self, embed_dim=768, num_heads=12, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5  # 1/sqrt(d_k)
        
        # QKV 一起投影（效率更高）
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(embed_dim, embed_dim)  # 输出投影
        self.proj_drop = nn.Dropout(proj_drop)
    
    def forward(self, x):
        B, N, C = x.shape  # batch, num_tokens, embed_dim
        
        # QKV: (B, N, 3*C) → (B, N, 3, num_heads, head_dim) → (3, B, num_heads, N, head_dim)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)  # 各 (B, num_heads, N, head_dim)
        
        # Attention: Q @ K^T / sqrt(d_k)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        # Attention @ V
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, C)
        
        # 输出投影
        x = self.proj_drop(self.proj(x))
        
        return x, attn  # 返回 attention map 用于可视化

# 测试
mhsa = MultiHeadSelfAttention(embed_dim=768, num_heads=12)
x = torch.randn(2, 197, 768)  # batch=2, 1 CLS + 196 patches
out, attn_weights = mhsa(x)
print(f"MHSA 输入: {x.shape}")
print(f"MHSA 输出: {out.shape}")
print(f"Attention 权重: {attn_weights.shape}  (batch, heads, tokens, tokens)")
print(f"\n每个 head 的维度: {768//12} = 768/12")
print(f"Attention 矩阵: 197×197 = {197*197} 个元素（O(n²) 复杂度！）")

In [ ]:
# ============================================================
# 实验 7.2: ViT Encoder Block (Pre-Norm)
# ============================================================

class FFN(nn.Module):
    """Feed-Forward Network with GELU activation"""
    def __init__(self, embed_dim=768, mlp_ratio=4., drop=0.):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.drop = nn.Dropout(drop)
    
    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x

class ViTBlock(nn.Module):
    """
    ViT Encoder Block (Pre-Norm 版本)
    
    Pre-Norm:  x' = x + MHSA(LN(x))
               x'' = x' + FFN(LN(x'))
    
    vs Post-Norm (原始 Transformer):
               x' = LN(x + MHSA(x))
               x'' = LN(x' + FFN(x'))
    """
    def __init__(self, embed_dim=768, num_heads=12, mlp_ratio=4., 
                 drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_drop, drop)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, mlp_ratio, drop)
    
    def forward(self, x):
        # Pre-Norm: LN 在 Attention/FFN 之前
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out              # 第一个残差连接
        x = x + self.ffn(self.norm2(x))  # 第二个残差连接
        return x, attn_weights

# 测试
block = ViTBlock(embed_dim=768, num_heads=12)
x = torch.randn(2, 197, 768)
out, attn = block(x)
print(f"ViT Block 输入: {x.shape}")
print(f"ViT Block 输出: {out.shape}")
print(f"\nFFN 隐藏层维度: {768 * 4} = 768 × 4")
print(f"FFN 参数量: {768*3072 + 3072 + 3072*768 + 768:,} = {(768*3072*2 + 3072 + 768)/1e6:.2f}M")

---
## 模块 8：手写完整 ViT（从零实现） (30min)

In [ ]:
# ============================================================
# 实验 8.1: 完整 Vision Transformer
# ============================================================

class VisionTransformer(nn.Module):
    """
    完整的 Vision Transformer 实现
    
    ViT-Base:  embed_dim=768,  depth=12, heads=12  → 86M params
    ViT-Large: embed_dim=1024, depth=24, heads=16  → 307M params
    ViT-Huge:  embed_dim=1280, depth=32, heads=16  → 632M params
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, embed_dim=768, depth=12, num_heads=12,
                 mlp_ratio=4., drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.embed_dim = embed_dim
        
        # ---- Patch Embedding ----
        self.patch_embed = PatchEmbedding_Conv(
            img_size, patch_size, in_channels, embed_dim
        )
        
        # ---- CLS Token ----
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        # ---- Position Embedding ----
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        
        self.pos_drop = nn.Dropout(drop_rate)
        
        # ---- Transformer Encoder ----
        self.blocks = nn.ModuleList([
            ViTBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop_rate)
            for _ in range(depth)
        ])
        
        # ---- 最终 LayerNorm ----
        self.norm = nn.LayerNorm(embed_dim)
        
        # ---- 分类头 ----
        self.head = nn.Linear(embed_dim, num_classes)
        
        # ---- 初始化 ----
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x, return_attention=False):
        B = x.shape[0]
        attentions = []
        
        # 1. Patch Embedding
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)
        
        # 2. 拼接 CLS Token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, 1+num_patches, embed_dim)
        
        # 3. 加位置编码
        x = self.pos_drop(x + self.pos_embed)
        
        # 4. Transformer Encoder
        for block in self.blocks:
            x, attn = block(x)
            if return_attention:
                attentions.append(attn)
        
        # 5. 取 CLS Token 的输出
        x = self.norm(x)
        cls_output = x[:, 0]  # (B, embed_dim)
        
        # 6. 分类
        logits = self.head(cls_output)
        
        if return_attention:
            return logits, attentions
        return logits

# ---- 工厂函数 ----
def vit_base(num_classes=1000, **kwargs):
    return VisionTransformer(
        embed_dim=768, depth=12, num_heads=12, num_classes=num_classes, **kwargs
    )

def vit_large(num_classes=1000, **kwargs):
    return VisionTransformer(
        embed_dim=1024, depth=24, num_heads=16, num_classes=num_classes, **kwargs
    )

# 测试
print("Vision Transformer 验证:")
print("=" * 50)

vit = vit_base(num_classes=1000)
x = torch.randn(2, 3, 224, 224)

with torch.no_grad():
    logits, attentions = vit(x, return_attention=True)

print(f"输入: {x.shape}")
print(f"输出: {logits.shape}")
print(f"Attention maps: {len(attentions)} 层, 每层 {attentions[0].shape}")

total_params = sum(p.numel() for p in vit.parameters())
print(f"\nViT-Base 总参数量: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"论文中 ViT-Base: ~86M 参数")

In [ ]:
# ============================================================
# 实验 8.2: ViT 各组件参数量分析
# ============================================================

vit = vit_base()

component_params = {
    'Patch Embedding': sum(p.numel() for p in vit.patch_embed.parameters()),
    'CLS Token': vit.cls_token.numel(),
    'Position Embed': vit.pos_embed.numel(),
    'Transformer Blocks': sum(p.numel() for p in vit.blocks.parameters()),
    'Final LayerNorm': sum(p.numel() for p in vit.norm.parameters()),
    'Classification Head': sum(p.numel() for p in vit.head.parameters()),
}

# 进一步分解 Transformer Block
block = vit.blocks[0]
block_detail = {
    '  LN1 + LN2': sum(p.numel() for p in block.norm1.parameters()) + 
                    sum(p.numel() for p in block.norm2.parameters()),
    '  MHSA (QKV+proj)': sum(p.numel() for p in block.attn.parameters()),
    '  FFN (2 Linear)': sum(p.numel() for p in block.ffn.parameters()),
}

print("ViT-Base 参数量分解:")
print("=" * 55)
total = sum(component_params.values())
for name, params in component_params.items():
    pct = params / total * 100
    print(f"  {name:25s}: {params:>12,} ({pct:5.1f}%)")
print(f"  {'Total':25s}: {total:>12,}")

print(f"\n单个 Transformer Block 分解 (×12):")
print("-" * 55)
block_total = sum(block_detail.values())
for name, params in block_detail.items():
    pct = params / block_total * 100
    print(f"  {name:25s}: {params:>12,} ({pct:5.1f}%)  ×12 = {params*12:>12,}")

print(f"\n💡 FFN 占了 Block 参数量的 ~2/3（因为 mlp_ratio=4 → 扩展 4 倍）")
print(f"   Transformer Blocks 总共占 ViT 参数量的 ~98%")

---
## 模块 9：CIFAR-10 训练对比 — ResNet vs ViT (40min)

### 适配 CIFAR-10 (32×32)

CIFAR-10 图像只有 32×32，需要调整:
- **ResNet**: 去掉 7×7 conv 和 maxpool，用 3×3 conv stem
- **ViT**: 使用 patch_size=4 (32/4=8, 得到 64 个 patch)

In [ ]:
# ============================================================
# 实验 9.1: CIFAR-10 数据加载 + 数据增强
# ============================================================

# 训练集数据增强
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# 测试集
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, 
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False,
                         num_workers=2, pin_memory=True)

classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"训练集: {len(train_dataset)} 张")
print(f"测试集: {len(test_dataset)} 张")
print(f"类别数: {len(classes)}")

In [ ]:
# ============================================================
# 实验 9.2: CIFAR-10 专用 ResNet
# ============================================================

class BasicBlock(nn.Module):
    """ResNet Basic Block (用于 ResNet-18/34)"""
    expansion = 1
    
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out)

class ResNet_CIFAR(nn.Module):
    """
    CIFAR-10 版 ResNet
    区别于 ImageNet 版:
    1. stem 用 3×3 conv (不是 7×7)
    2. 没有 MaxPool
    3. 通道数更小
    """
    def __init__(self, block, layers, num_classes=10):
        super().__init__()
        self.in_planes = 64
        
        # CIFAR stem: 3x3 conv (不是 7x7, 也没有 maxpool)
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.layer1 = self._make_layer(block, 64,  layers[0], stride=1)  # 32×32
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)  # 16×16
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)  #  8×8
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)  #  4×4
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        
        self._init_weights()
    
    def _make_layer(self, block, planes, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes * block.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion)
            )
        layers = [block(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_planes, planes))
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        return self.fc(x)

def resnet18_cifar(num_classes=10):
    return ResNet_CIFAR(BasicBlock, [2, 2, 2, 2], num_classes)

# 测试
model = resnet18_cifar()
x = torch.randn(2, 3, 32, 32)
y = model(x)
params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 (CIFAR): {x.shape} → {y.shape}, {params/1e6:.2f}M params")

In [ ]:
# ============================================================
# 实验 9.3: CIFAR-10 专用 ViT (小模型)
# ============================================================

class ViT_CIFAR(nn.Module):
    """
    CIFAR-10 版 ViT (轻量版)
    img_size=32, patch_size=4 → 64 patches
    使用较小的 embed_dim 和 depth 以匹配 ResNet 的参数量
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=10, embed_dim=256, depth=6, num_heads=8,
                 mlp_ratio=4., drop_rate=0.1):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2  # 64
        self.embed_dim = embed_dim
        
        # Patch Embedding (用 Conv)
        self.patch_embed = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        
        # CLS Token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        # Position Embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        
        self.pos_drop = nn.Dropout(drop_rate)
        
        # Transformer Encoder
        self.blocks = nn.ModuleList([
            ViTBlock(embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop=0.)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x, return_attention=False):
        B = x.shape[0]
        attentions = []
        
        # Patch Embedding
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        
        # CLS + Position
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        
        # Transformer Encoder
        for block in self.blocks:
            x, attn = block(x)
            if return_attention:
                attentions.append(attn)
        
        # CLS → classification
        x = self.norm(x)[:, 0]
        logits = self.head(x)
        
        if return_attention:
            return logits, attentions
        return logits

# 测试
vit_cifar = ViT_CIFAR()
x = torch.randn(2, 3, 32, 32)
y = vit_cifar(x)
params = sum(p.numel() for p in vit_cifar.parameters())
print(f"ViT-CIFAR: {x.shape} → {y.shape}, {params/1e6:.2f}M params")
print(f"Patches: (32/4)² = 64")
print(f"序列长度: 64 + 1 (CLS) = 65")

In [ ]:
# ============================================================
# 实验 9.4: 通用训练 + 评估函数
# ============================================================

def train_one_epoch(model, train_loader, optimizer, scheduler, device, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        
        # 梯度裁剪（对 ViT 很重要）
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    
    return total_loss / total, 100. * correct / total

@torch.no_grad()
def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        total_loss += F.cross_entropy(output, target, reduction='sum').item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    
    return total_loss / total, 100. * correct / total

def train_model(model, model_name, train_loader, test_loader, device,
                epochs=30, lr=1e-3, weight_decay=5e-4):
    """
    完整训练流程
    """
    model = model.to(device)
    
    # 优化器: ResNet 用 SGD+momentum, ViT 用 AdamW
    if 'resnet' in model_name.lower():
        optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        total_steps = epochs * len(train_loader)
        warmup_steps = 5 * len(train_loader)  # 5 epochs warmup
        
        def lr_lambda(step):
            if step < warmup_steps:
                return step / warmup_steps
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return 0.5 * (1 + math.cos(math.pi * progress))
        
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params:,} ({params/1e6:.2f}M)")
    print(f"Optimizer: {type(optimizer).__name__}, LR: {lr if 'vit' in model_name.lower() else 0.1}")
    print(f"{'='*60}")
    
    start_time = time.time()
    best_acc = 0
    
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, device, epoch
        )
        test_loss, test_acc = evaluate(model, test_loader, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        if test_acc > best_acc:
            best_acc = test_acc
        
        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - start_time
            print(f"  Epoch {epoch:3d}/{epochs}: "
                  f"Train Loss={train_loss:.4f} Acc={train_acc:.1f}% | "
                  f"Test Loss={test_loss:.4f} Acc={test_acc:.1f}% | "
                  f"Best={best_acc:.1f}% | "
                  f"Time={elapsed:.0f}s")
    
    total_time = time.time() - start_time
    print(f"\n  ✅ {model_name} 完成! Best Test Acc: {best_acc:.1f}%, Time: {total_time:.0f}s")
    
    return history, best_acc, total_time

print("训练函数定义完毕 ✅")

In [ ]:
# ============================================================
# 实验 9.5: 训练 ResNet-18 和 ViT
# ============================================================
# ⚠️ 如果没有 GPU，可以减少 epochs 到 10-15

EPOCHS = 30  # GPU: 30, CPU: 10

# 1. 训练 ResNet-18
resnet_model = resnet18_cifar(num_classes=10)
resnet_history, resnet_best, resnet_time = train_model(
    resnet_model, 'ResNet-18', train_loader, test_loader, device,
    epochs=EPOCHS, lr=0.1, weight_decay=5e-4
)

# 2. 训练 ViT
vit_model = ViT_CIFAR(
    img_size=32, patch_size=4, embed_dim=256, depth=6, num_heads=8,
    mlp_ratio=4., drop_rate=0.1
)
vit_history, vit_best, vit_time = train_model(
    vit_model, 'ViT-CIFAR', train_loader, test_loader, device,
    epochs=EPOCHS, lr=1e-3, weight_decay=0.05
)

In [ ]:
# ============================================================
# 实验 9.6: 训练曲线对比
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(resnet_history['train_loss']) + 1)

# Loss 曲线
axes[0].plot(epochs_range, resnet_history['train_loss'], 'b-', label='ResNet Train', alpha=0.7)
axes[0].plot(epochs_range, resnet_history['test_loss'], 'b--', label='ResNet Test')
axes[0].plot(epochs_range, vit_history['train_loss'], 'r-', label='ViT Train', alpha=0.7)
axes[0].plot(epochs_range, vit_history['test_loss'], 'r--', label='ViT Test')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 曲线
axes[1].plot(epochs_range, resnet_history['train_acc'], 'b-', label='ResNet Train', alpha=0.7)
axes[1].plot(epochs_range, resnet_history['test_acc'], 'b--', label='ResNet Test')
axes[1].plot(epochs_range, vit_history['train_acc'], 'r-', label='ViT Train', alpha=0.7)
axes[1].plot(epochs_range, vit_history['test_acc'], 'r--', label='ViT Test')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 对比总结
resnet_params = sum(p.numel() for p in resnet_model.parameters())
vit_params = sum(p.numel() for p in vit_model.parameters())

comparison = {
    'Model': ['ResNet-18', 'ViT-CIFAR'],
    'Params': [f'{resnet_params/1e6:.2f}M', f'{vit_params/1e6:.2f}M'],
    'Best Acc': [f'{resnet_best:.1f}%', f'{vit_best:.1f}%'],
    'Time': [f'{resnet_time:.0f}s', f'{vit_time:.0f}s'],
}

# 表格展示
axes[2].axis('off')
table_data = [[comparison['Model'][i], comparison['Params'][i], 
               comparison['Best Acc'][i], comparison['Time'][i]] for i in range(2)]
table = axes[2].table(
    cellText=table_data,
    colLabels=['Model', 'Params', 'Best Acc', 'Time'],
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2)
axes[2].set_title('Comparison Summary', fontsize=14, pad=20)

plt.tight_layout()
plt.show()

print("\n📊 分析:")
print("  • ResNet 在小数据集上通常优于 ViT（归纳偏置的优势）")
print("  • ViT 缺乏卷积的局部性偏置，需要更多数据")
print("  • 在 ImageNet-21k/JFT-300M 上预训练后，ViT 可以超越 ResNet")
print("  • 这就是为什么 CLIP/DINOv2 等大规模预训练对 ViT 如此重要")

---
## 模块 10：可视化分析 + 总结 (30min)

In [ ]:
# ============================================================
# 实验 10.1: ResNet 特征图可视化
# ============================================================

def visualize_resnet_features(model, test_loader, device):
    """可视化 ResNet 各 stage 的特征图"""
    model.eval()
    features = {}
    
    def make_hook(name):
        def hook(m, i, o):
            features[name] = o.detach().cpu()
        return hook
    
    hooks = []
    hooks.append(model.layer1.register_forward_hook(make_hook('Stage 1')))
    hooks.append(model.layer2.register_forward_hook(make_hook('Stage 2')))
    hooks.append(model.layer3.register_forward_hook(make_hook('Stage 3')))
    hooks.append(model.layer4.register_forward_hook(make_hook('Stage 4')))
    
    # 取一个 batch
    data, targets = next(iter(test_loader))
    data = data[:1].to(device)  # 只取一张图
    
    with torch.no_grad():
        _ = model(data)
    
    for h in hooks:
        h.remove()
    
    # 可视化
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # 原图
    img = data[0].cpu()
    # 反归一化
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).reshape(3,1,1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).reshape(3,1,1)
    img = img * std + mean
    img = img.clamp(0, 1)
    
    axes[0, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[0, 0].set_title(f'Input ({classes[targets[0]]})', fontsize=11)
    axes[0, 0].axis('off')
    
    # 各 stage 特征图（取前 8 个通道的平均）
    for idx, (name, feat) in enumerate(features.items()):
        ax_row = (idx + 1) // 4
        ax_col = (idx + 1) % 4
        # 平均前几个通道
        feat_map = feat[0, :8].mean(dim=0).numpy()
        axes[ax_row, ax_col].imshow(feat_map, cmap='viridis')
        h, w = feat_map.shape
        axes[ax_row, ax_col].set_title(f'{name}: {feat[0].shape[0]}ch × {h}×{w}', fontsize=11)
        axes[ax_row, ax_col].axis('off')
    
    # 隐藏多余的子图
    for i in range(5, 8):
        axes[i // 4, i % 4].axis('off')
    
    plt.suptitle('ResNet Feature Maps at Different Stages', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_resnet_features(resnet_model, test_loader, device)

In [ ]:
# ============================================================
# 实验 10.2: ViT Attention Map 可视化
# ============================================================

def visualize_vit_attention(model, test_loader, device, patch_size=4, img_size=32):
    """可视化 ViT 各层的 Attention Map"""
    model.eval()
    
    data, targets = next(iter(test_loader))
    data = data[:1].to(device)
    
    with torch.no_grad():
        _, attentions = model(data, return_attention=True)
    
    num_patches_h = img_size // patch_size  # 8
    
    # 选择几层可视化
    layers_to_show = [0, len(attentions)//2, len(attentions)-1]  # 第一、中间、最后
    
    fig, axes = plt.subplots(len(layers_to_show), 5, figsize=(16, 3*len(layers_to_show)))
    
    for row, layer_idx in enumerate(layers_to_show):
        attn = attentions[layer_idx][0].cpu()  # (heads, N, N)
        
        # CLS token 对所有 patch 的 attention
        cls_attn = attn[:, 0, 1:]  # (heads, num_patches)
        
        # 展示前 4 个 head + 平均
        for col in range(min(4, attn.shape[0])):
            attn_map = cls_attn[col].reshape(num_patches_h, num_patches_h).numpy()
            axes[row, col].imshow(attn_map, cmap='hot', interpolation='bilinear')
            if row == 0:
                axes[row, col].set_title(f'Head {col}', fontsize=11)
            axes[row, col].axis('off')
        
        # 平均所有 head
        avg_attn = cls_attn.mean(dim=0).reshape(num_patches_h, num_patches_h).numpy()
        axes[row, 4].imshow(avg_attn, cmap='hot', interpolation='bilinear')
        if row == 0:
            axes[row, 4].set_title('Average', fontsize=11)
        axes[row, 4].axis('off')
        
        axes[row, 0].set_ylabel(f'Layer {layer_idx}', fontsize=12, rotation=0, labelpad=40)
    
    plt.suptitle(f'ViT CLS Token Attention Maps (input: {classes[targets[0]]})', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("💡 观察:")
    print("  • 不同 head 关注不同区域（类似不同的滤波器）")
    print("  • 浅层 attention 较均匀，深层更聚焦于语义区域")
    print("  • CLS token 从所有 patch 聚合信息")

visualize_vit_attention(vit_model, test_loader, device)

In [ ]:
# ============================================================
# 实验 10.3: ResNet vs ViT 归纳偏置对比
# ============================================================

print("=" * 70)
print("ResNet vs ViT: 归纳偏置 (Inductive Bias) 深度对比")
print("=" * 70)

comparison_table = """
┌──────────────────┬─────────────────────────┬────────────────────────────┐
│     特性          │ ResNet (CNN)             │ ViT (Transformer)          │
├──────────────────┼─────────────────────────┼────────────────────────────┤
│ 局部性           │ ✅ 卷积核限制感受野       │ ❌ 全局注意力（第一层就全局）│
│ 平移等变性       │ ✅ 权重共享天然保证       │ ❌ 需要数据增强/位置编码学   │
│ 层级结构         │ ✅ 自然从低级到高级特征   │ ❌ 每层都是全局交互         │
│ 尺度不变性       │ ❌ 固定感受野增长         │ ✅ 注意力可动态调整         │
│ 参数效率(小数据) │ ✅ 归纳偏置减少搜索空间   │ ❌ 需要大量数据学习结构     │
│ 扩展性(大数据)   │ ❌ 偏置可能限制上限       │ ✅ 更灵活，上限更高         │
│ 长距离依赖       │ ❌ 需要很多层             │ ✅ 单层即可                 │
│ 训练稳定性       │ ✅ BN + 残差，较稳定      │ ❌ 需要 warmup + 梯度裁剪   │
└──────────────────┴─────────────────────────┴────────────────────────────┘
"""
print(comparison_table)

print("📊 数据量 vs 性能:")
print("  • < 10K: ResNet >> ViT  (ViT 严重过拟合)")
print("  • ~50K (CIFAR-10): ResNet > ViT  (ViT 还是差一些)")
print("  • ~1M (ImageNet-1K): ResNet ≈ ViT  (差不多)")
print("  • ~14M (ImageNet-21K): ViT > ResNet  (开始超越)")
print("  • ~300M+ (JFT-300M): ViT >> ResNet  (大幅领先)")
print("\n💡 结论: 归纳偏置是双刃剑 — 小数据时是优势，大数据时是限制")

In [ ]:
# ============================================================
# 实验 10.4: 推理速度 + 显存对比
# ============================================================

def benchmark_model(model, input_shape, device, name, n_warmup=10, n_iter=50):
    """推理速度基准测试"""
    model = model.to(device)
    model.eval()
    x = torch.randn(*input_shape).to(device)
    
    # 预热
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)
    
    # 记录显存
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    
    # 计时
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.time()
    with torch.no_grad():
        for _ in range(n_iter):
            _ = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = (time.time() - start) / n_iter
    throughput = input_shape[0] / elapsed
    
    mem = 0
    if device.type == 'cuda':
        mem = torch.cuda.max_memory_allocated() / 1024**2
    
    params = sum(p.numel() for p in model.parameters())
    
    return {
        'name': name,
        'params': params,
        'latency_ms': elapsed * 1000,
        'throughput': throughput,
        'memory_mb': mem,
    }

# 对比
results = []
batch_size = 32

resnet_bench = benchmark_model(
    resnet18_cifar(), (batch_size, 3, 32, 32), device, 'ResNet-18'
)
results.append(resnet_bench)

vit_bench = benchmark_model(
    ViT_CIFAR(), (batch_size, 3, 32, 32), device, 'ViT-CIFAR'
)
results.append(vit_bench)

print(f"\n{'Model':15s} {'Params':>10s} {'Latency':>10s} {'Throughput':>12s} {'Memory':>10s}")
print("-" * 60)
for r in results:
    mem_str = f"{r['memory_mb']:.1f}MB" if r['memory_mb'] > 0 else "N/A"
    print(f"{r['name']:15s} {r['params']/1e6:>9.2f}M {r['latency_ms']:>8.1f}ms "
          f"{r['throughput']:>10.0f}img/s {mem_str:>10s}")

In [ ]:
# ============================================================
# 实验 10.5: Attention 复杂度 O(n²d) 的实验验证
# ============================================================

# 测量不同序列长度下 Self-Attention 的时间
embed_dim = 256
num_heads = 8
attn_module = MultiHeadSelfAttention(embed_dim, num_heads).to(device)
attn_module.eval()

seq_lengths = [16, 32, 64, 128, 256, 512]
times = []

for seq_len in seq_lengths:
    x = torch.randn(4, seq_len, embed_dim).to(device)
    
    # 预热
    with torch.no_grad():
        for _ in range(5):
            _ = attn_module(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.time()
    with torch.no_grad():
        for _ in range(20):
            _ = attn_module(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = (time.time() - start) / 20
    times.append(elapsed * 1000)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, times, 'bo-', linewidth=2, markersize=8, label='Measured')
# 拟合 O(n²) 曲线
scale = times[-1] / (seq_lengths[-1] ** 2)
fit_times = [scale * n**2 for n in seq_lengths]
axes[0].plot(seq_lengths, fit_times, 'r--', linewidth=2, label='O(n²) fit')
axes[0].set_xlabel('Sequence Length (n)')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Self-Attention Time Complexity')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# log-log 图
axes[1].loglog(seq_lengths, times, 'bo-', linewidth=2, markersize=8, label='Measured')
axes[1].loglog(seq_lengths, fit_times, 'r--', linewidth=2, label='O(n²) reference')
axes[1].set_xlabel('Sequence Length (n) [log]')
axes[1].set_ylabel('Time (ms) [log]')
axes[1].set_title('Log-Log Scale (slope ≈ 2 → O(n²))')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 计算斜率
log_n = np.log(seq_lengths)
log_t = np.log(times)
slope = np.polyfit(log_n, log_t, 1)[0]
axes[1].text(0.05, 0.95, f'Slope = {slope:.2f}', transform=axes[1].transAxes,
             fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"\n斜率 = {slope:.2f} (理论值 = 2.0 for O(n²))")
print(f"\n💡 这就是为什么 ViT 处理高分辨率图像很贵:")
print(f"   224×224, patch=16 → 196 tokens → Attention: 196² = {196**2:,} 运算")
print(f"   512×512, patch=16 → 1024 tokens → Attention: 1024² = {1024**2:,} 运算")
print(f"   → 分辨率翻倍，Attention 计算量增加 ~27 倍！")

---
## 📝 Day6 总结

### 今日收获

**CNN 基础**:
- 卷积 = 局部连接 + 权重共享，参数量极大减少
- 输出尺寸公式: `H_out = floor((H + 2p - d(k-1) - 1) / s + 1)`
- 感受野: 三层 3×3 = 7×7 感受野，比一层 7×7 更高效

**ResNet**:
- 残差连接解决退化问题: `H(x) = F(x) + x`
- 梯度直达: `∂L/∂x = ∂L/∂H · (1 + ∂F/∂x)`
- Bottleneck: 1×1降维 → 3×3空间 → 1×1升维
- 手写 ResNet-50 与 torchvision 完全一致 ✅

**ViT**:
- 图像 → Patch → Linear Projection → + CLS Token → + Position Embedding → Transformer Encoder
- Pre-Norm: LN 在 Attention/FFN 之前
- Patch Embedding 可用 Conv2d 高效实现
- O(n²d) 复杂度限制了高分辨率应用

**核心对比**:
- ResNet 有强归纳偏置 → 小数据友好
- ViT 更灵活 → 大数据下上限更高
- 现代趋势: ViT + 大规模预训练 (CLIP/DINOv2)

### 面试要点
1. 为什么 ResNet 能训练到 152 层？→ 残差连接保证梯度流
2. ViT 为什么需要大数据？→ 缺乏局部性归纳偏置
3. Patch Embedding 两种实现？→ 手动切片+Linear / Conv2d
4. Pre-Norm vs Post-Norm？→ Pre-Norm 训练更稳定
5. ViT 的 CLS Token 作用？→ 聚合全局信息用于分类

### 明日预告
- W1 Day7: 复盘 + ConSE 工程深挖